# 7b. XRO forecasts of the 2015/16 and 2026/27 El Niños

Second of the four XRO notebooks (7a validation → **7b events** → 7c E3SM state → 7d recharge).
Uses observations only. Read 7a first: it gives XRO's typical out-of-sample skill, how much it
underpredicts strong events from each initial month, and how overconfident its stochastic
ensemble is.

For each event, XRO is detrended and fitted on the years before the event (`xc.EVENTS[...]["train"]`,
so the forecasts are out of sample) and run from the observed state of the month before each
hindcast start: an **observationally initialized reduced-order benchmark**, not a forecast from the
same initial state as E3SM (E3SM starts from the 1 Feb/May/Aug/Nov coupled reanalysis state; its
lead 0 is the start month's mean). Leads here follow the hindcasts: lead 0 = start month, lead −1
= XRO's initial month.

Sections:

1. Niño3.4 and *h* forecasts from every start (deterministic, stochastic 5-95%, and the linear-*h*
   detrending alternative).
2. Peak-season (DJF) summary.
3. Sensitivity of the peak-season Niño3.4 to individual initial-state components (removal
   experiments; XRO is nonlinear, so these are sensitivities, not an additive attribution).
4. Sensitivity to the initial recharge: the peak-season Niño3.4 when only the initial *h* is
   shifted by δ*h*, spanning the E3SM-EN4 initial differences found in 7c (−0.3 to −0.7 GJ m⁻²).

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from datetime import datetime

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from IPython.display import display

from esp_lab.utils.netcdf_utils import atomic_to_netcdf

from util import rtenso_case as rc
from util import xro_case as xc
from util import pubfig as pf

pf.use_style()

### Run controls

In [ ]:
NOTEBOOK = "7b_xro_events"
derivation_mode = "auto"
# Detrending options; the first is the control (xc.XRO_SETUP["detrend_order"]), the second is shown as a sensitivity
TREND_OPTIONS = {"quadratic h": {"default": 1, "WWV": 2}, "linear h": {"default": 1, "WWV": 1}}
CONTROL = next(iter(TREND_OPTIONS))
assert TREND_OPTIONS[CONTROL] == xc.XRO_SETUP["detrend_order"]
DELTA_H = np.round(np.arange(-1.0, 0.61, 0.1), 2)   # initial-h shifts (GJ m-2) for section 4
START_COLORS = {2: "#0072b2", 5: "#009e73", 8: "#e69f00", 11: "#cc79a7"}
LO, HI = xc.XRO_SETUP["spread"]
OUTDIR = rc.output_dir(NOTEBOOK, "data")
print("figures ->", rc.figure_dir(NOTEBOOK))

In [ ]:
X = xc.obs_state(mode=derivation_mode)
FITS = {(year, name): xc.fit_xro(X, event["train"], orders=orders)
        for year, event in xc.EVENTS.items() for name, orders in TREND_OPTIONS.items()}
FC, FC_ALT, SENS, FC_CAL = {}, {}, {}, {}
for year, event in xc.EVENTS.items():
    for init in event["init_months"]:
        fc = xc.event_forecast(FITS[(year, CONTROL)], year, init)
        if fc is None:
            ic = xc.add_months(year, init, -1)
            print(f"[{year} init{init:02d}] no fully observed initial state for {ic[0]}-{ic[1]:02d}")
            continue
        FC[(year, init)] = fc
        FC_CAL[(year, init)] = xc.calibrated_stoch(X, year, init, fc["stoch"], mode=derivation_mode)
        FC_ALT[(year, init)] = xc.event_forecast(FITS[(year, "linear h")], year, init, stochastic=False)
        SENS[(year, init)] = xc.removal_sensitivity(FITS[(year, CONTROL)], year, init)
        X0 = xc.initial_state(FITS[(year, CONTROL)], year, init)
        out = xr.Dataset({f"{k}_{v}": fc[k][v] for k in ("det", "det_dt", "stoch") for v in xc.XRO_VARS})
        atomic_to_netcdf(out.drop_vars("time"), OUTDIR / f"XRO_init{init:02d}_y{year}_train{event['train'][0]}-{event['train'][1]}_fcst_mon.nc")
        print(f"[{year} init{init:02d}] initial state {xc.MONTH_NAMES[xc.add_months(year, init, -1)[1]]}: "
              f"Niño3.4 {float(X0['Nino34']):+.2f} °C, h {float(X0['WWV']):+.2f} GJ m-2 (detrended)")

## 1. Niño3.4 and *h* forecasts

Black: observed (HadISST 1.1, EN4). Purple dashed: deterministic XRO; shading: stochastic
`LO`-`HI`% range (7a section 7: this range is too narrow, about 60-80% coverage instead of 90%).
Thin dotted: deterministic XRO with a linear *h* trend. Grey: the peak season. The vertical line
marks the start.
Thin solid purple lines (Niño3.4 only): the `LO`-`HI`% range after the cross-validated spread
calibration of 7a section 8 (factors from validation forecasts verified before XRO's initial month;
the ensemble mean is unchanged). *h* is not calibrated: the same correction made its intervals too
wide in 7a.

In [ ]:
def to_dt(times):
    return [datetime(t.year, t.month, 15) for t in np.atleast_1d(times)]


def plot_event(year):
    event = xc.EVENTS[year]
    inits = [i for i in event["init_months"] if (year, i) in FC]
    fig, axes = plt.subplots(len(inits), 2, figsize=(pf.WIDTH["double"], 1.5 * len(inits) + 0.5), sharex=True)
    axes = np.atleast_2d(axes)
    obs = X.sel(time=slice(f"{year - 1}-10", f"{year + 2}-03"))
    (m0y, m0m), (m1y, m1m) = event["peak_season"]["months"][0], event["peak_season"]["months"][-1]
    for r, init in enumerate(inits):
        fc = FC[(year, init)]
        for c, var in enumerate(["Nino34", "WWV"]):
            ax = axes[r, c]
            ax.axvspan(datetime(m0y, m0m, 1), datetime(m1y, m1m, 28), color="0.9", lw=0, zorder=0)
            x = to_dt(fc["det"]["time"].values)
            ax.fill_between(x, fc["stoch"][var].quantile(LO / 100, "member"), fc["stoch"][var].quantile(HI / 100, "member"),
                            color=xc.XRO_STYLE["color"], alpha=0.2, lw=0, label=f"XRO stochastic {LO}-{HI}%")
            if var in xc.CALIBRATION["vars"]:
                cal = FC_CAL[(year, init)][var]
                for q in (LO, HI):
                    ax.plot(x, cal.quantile(q / 100, "member"), color=xc.XRO_STYLE["color"], lw=0.6, ls="-",
                            label=f"XRO calibrated {LO}-{HI}%" if q == LO else None)
            ax.plot(x, fc["det"][var], color=xc.XRO_STYLE["color"], ls="--", label="XRO deterministic")
            ax.plot(to_dt(FC_ALT[(year, init)]["det"]["time"].values), FC_ALT[(year, init)]["det"][var],
                    color=xc.XRO_STYLE["color"], ls=":", lw=0.9, label="XRO, linear h trend")
            ax.plot(to_dt(obs["time"].values), obs[var], color="k", lw=1.4, label="Observed")
            ax.axvline(datetime(year, init, 1), color="0.3", lw=0.8, ls="-", label="Start")
            pf.zero_line(ax)
            ax.set_ylabel(xc.UNITS[var])
            pf.panel_label(ax, 2 * r + c, f"{xc.short(var)}, 1 {xc.MONTH_NAMES[init]} {year} start")
            ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=(1, 7)))
            ax.xaxis.set_minor_locator(mdates.MonthLocator())
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
            ax.set_xlim(datetime(year - 1, 12, 1), datetime(year + 2, 2, 28))
    handles, labels = axes[0, 0].get_legend_handles_labels()
    pf.legend_below(fig, handles, labels, ncol=6)
    return fig


for year in xc.EVENTS:
    fig = plot_event(year)
    pf.save(fig, NOTEBOOK, "xro", f"y{year}", "nino34_h_forecasts")
    plt.show()

## 2. Peak-season summary

DJF means: observed (NaN until observed), deterministic XRO (control and linear-*h* trend), and the
stochastic mean with its `LO`-`HI`% range. Compare the deterministic values with 7a section 6: from
the Feb and May starts a skilful XRO forecast of a strong event is typically only 30-70% of the
observed amplitude.
"calibrated" columns: Niño3.4 range from the calibrated monthly members. Monthly calibration is
only approximately right for seasonal means (7a section 8, DJF table).

In [ ]:
rows = []
for (year, init), fc in FC.items():
    months = xc.EVENTS[year]["peak_season"]["months"]
    for var in ("Nino34", "WWV"):
        stoch = rc.target_mean(fc["stoch"][var], months)
        rows.append({"event": xc.EVENTS[year]["label"], "start": f"1 {xc.MONTH_NAMES[init]} {year}", "variable": xc.short(var),
                     "observed": round(float(rc.obs_target_mean(X[var], months)), 2),
                     "XRO det": round(float(rc.target_mean(fc["det"][var], months)), 2),
                     "XRO det (linear h)": round(float(rc.target_mean(FC_ALT[(year, init)]["det"][var], months)), 2),
                     "XRO stoch mean": round(float(stoch.mean("member")), 2),
                     f"stoch {LO}%": round(float(stoch.quantile(LO / 100, "member")), 2),
                     f"stoch {HI}%": round(float(stoch.quantile(HI / 100, "member")), 2),
                     **({f"calibrated {q}%": round(float(rc.target_mean(FC_CAL[(year, init)][var], months).quantile(q / 100, "member")), 2)
                         for q in (LO, HI)} if var in xc.CALIBRATION["vars"] else {})})
peak = pd.DataFrame(rows)
peak.to_csv(OUTDIR / "XRO_event_peak_summary.csv", index=False)
display(peak[peak["variable"] == "Niño3.4"].set_index(["event", "start"]).drop(columns="variable"))

## 3. Peak-season Niño3.4 sensitivity to individual initial-state components

Each bar is the detrended deterministic DJF Niño3.4 minus the same forecast started with one
variable's initial anomaly set to zero; "other modes" zeroes all eight non-ENSO modes together.
Diamonds: full detrended forecast. Open circles: the sum of the ten single-variable sensitivities;
its distance from the diamond measures the nonlinear interaction that removal experiments cannot
assign to one variable. These are XRO's sensitivities, not a unique causal decomposition, and only
say which observed precursors XRO relies on.

In [ ]:
names = xc.XRO_VARS + ["other modes"]
years = list(xc.EVENTS)
fig, axes = plt.subplots(len(years), 1, figsize=(pf.WIDTH["double"], 2.2 * len(years)), sharex=True)
sens_rows = []
for p, (ax, year) in enumerate(zip(np.atleast_1d(axes), years)):
    event = xc.EVENTS[year]
    months = event["peak_season"]["months"]
    inits = [i for i in event["init_months"] if (year, i) in SENS]
    width = 0.8 / max(len(inits), 1)
    for k, init in enumerate(inits):
        contrib = rc.target_mean(SENS[(year, init)]["Nino34"], months)
        full = float(rc.target_mean(FC[(year, init)]["det_dt"]["Nino34"], months))
        vals = [float(contrib.sel(removed=n)) for n in names]
        xpos = np.arange(len(names)) + (k - (len(inits) - 1) / 2) * width
        ax.bar(xpos, vals, width=width, color=START_COLORS[init], edgecolor="k", lw=0.3,
               label=f"1 {xc.MONTH_NAMES[init]} start")
        xf = len(names) + (k - (len(inits) - 1) / 2) * width
        ax.plot(xf, full, marker="D", ms=4, color=START_COLORS[init], mec="k", mew=0.4)
        ax.plot(xf, sum(vals[:len(xc.XRO_VARS)]), marker="o", ms=4, mfc="none", mec=START_COLORS[init], mew=0.8)
        sens_rows.append({"event": event["label"], "start": xc.MONTH_NAMES[init], **dict(zip(names, np.round(vals, 2))),
                          "sum of single": round(sum(vals[:len(xc.XRO_VARS)]), 2), "full (detrended)": round(full, 2)})
    pf.zero_line(ax)
    ax.axvline(len(names) - 1.5, color="0.6", lw=0.6, ls=":")
    ax.axvline(len(names) - 0.5, color="0.6", lw=0.6, ls=":")
    ax.set_ylabel(f"Δ {event['peak_season']['label']}\nNiño3.4 (°C)")
    ax.grid(axis="y")
    pf.panel_label(ax, p, event["label"])
from matplotlib.lines import Line2D
handles, labels = np.atleast_1d(axes)[0].get_legend_handles_labels()
handles += [Line2D([], [], marker="D", ls="none", color="0.4", mec="k", mew=0.4, ms=4),
            Line2D([], [], marker="o", ls="none", mfc="none", mec="0.4", ms=4)]
labels += ["full forecast (detrended)", "sum of single-variable sensitivities"]
pf.legend_below(fig, handles, labels, ncol=6)
ax.set_xticks(range(len(names) + 1), [xc.short(n) for n in names] + ["full"], rotation=30, ha="right")
pf.save(fig, NOTEBOOK, "xro", "peak_nino34_ic_sensitivity")
plt.show()
sens_table = pd.DataFrame(sens_rows)
sens_table.to_csv(OUTDIR / "XRO_peak_nino34_ic_sensitivity.csv", index=False)
display(sens_table.set_index(["event", "start"]))

## 4. Sensitivity to the initial recharge

Peak-season Niño3.4 (deterministic, trend added back) when only the initial *h* is shifted by δ*h*
(all other initial values observed). The slope is XRO's peak-season response per GJ m⁻² of initial
recharge. 7d uses it with the E3SM-EN4 initial *h* differences to ask how much of the E3SM-XRO
difference a different initial recharge alone could explain. Dashed: observed DJF Niño3.4 (2015/16).

In [ ]:
fig, axes = plt.subplots(1, len(years), figsize=(pf.WIDTH["double"], 2.3), sharey=True)
dh_rows = []
for p, (ax, year) in enumerate(zip(np.atleast_1d(axes), years)):
    event = xc.EVENTS[year]
    months = event["peak_season"]["months"]
    for init in event["init_months"]:
        if (year, init) not in FC:
            continue
        resp = [float(rc.target_mean(xc.perturbed_forecast(FITS[(year, CONTROL)], year, init, {"WWV": d})["Nino34"], months))
                for d in DELTA_H]
        slope = np.polyfit(DELTA_H, resp, 1)[0]
        ax.plot(DELTA_H, resp, color=START_COLORS[init], marker="o", ms=2.5,
                label=f"1 {xc.MONTH_NAMES[init]} ({slope:+.2f} °C per GJ m$^{{-2}}$)")
        dh_rows += [{"event": event["label"], "start": xc.MONTH_NAMES[init], "delta_h": d, "DJF_Nino34": round(v, 3)}
                    for d, v in zip(DELTA_H, resp)]
    obs_peak = float(rc.obs_target_mean(X["Nino34"], months))
    if np.isfinite(obs_peak):
        ax.axhline(obs_peak, color="k", ls="--", lw=0.8, label="Observed")
    ax.axvline(0, color="0.5", lw=0.6)
    ax.axvspan(-0.7, -0.3, color="0.9", lw=0, zorder=0)
    ax.set_xlabel("Initial δh (GJ m$^{-2}$)")
    ax.grid(True)
    pf.panel_label(ax, p, f"{event['label']}: {event['peak_season']['label']}")
    ax.legend(loc="upper left", fontsize=6)
np.atleast_1d(axes)[0].set_ylabel("Peak-season Niño3.4 (°C)")
pf.save(fig, NOTEBOOK, "xro", "peak_nino34_initial_h_sensitivity")
plt.show()
pd.DataFrame(dh_rows).to_csv(OUTDIR / "XRO_peak_nino34_initial_h_sensitivity.csv", index=False)